In [32]:
import numpy as np 
import matplotlib.pyplot as plt
from matplotlib import rc
from ellipse import LsqEllipse
from math import comb
from matplotlib.patches import Ellipse
import scipy.optimize as sco
from scipy.optimize import fmin
from scipy.optimize import curve_fit
from scipy.optimize import root_scalar
import scipy
import json
import glob
import os
import allantools
from tqdm.notebook import trange, tqdm
import warnings
from clock_sim import *
import pickle


f0 = 429e12
h_const = 6.626e-34
G = 6.67e-11
M0 = 2e30
c = 3e8

def chirp_constant(Mc):
    return (96/5) * np.pi**(8/3) * (G * Mc / c**3)**(5/3)

def f_t(t, f_0, Mc):
    k = chirp_constant(Mc)
    return (f_0**(-8/3) - (8/3) * k * t)**(-3/8)

def time_to_frequency(f, f_0, Mc):

    k = chirp_constant(Mc)
    return (3 / (8 * k)) * (f_0**(-8/3) - f**(-8/3))


def h_chirped(t,H,f,phi):
    return H*np.sin(2*np.pi*f*t+phi)

def signal_chirped(t0,f_chirp,H= 1e-20,phi = 0,spin_echo = True,fixed_T = False, T = 100, optimized_T = True,d = 1e10):
    if fixed_T == False:
        T = 1/f_chirp(t0)
    if optimized_T:
        T = find_T_minimize(t0,f_chirp)
        
    if spin_echo == False:
        T = find_T_minimize_half_cycle(t0,f_chirp)
    ts = np.linspace(t0,t0+T,40000)
    dt = ts[1]-ts[0]
    s = h_chirped(ts,H,f_chirp(ts),phi)-h_chirped(ts-d/c,H,f_chirp(ts-d/c),phi)
    s_bar = np.sum((1/T * s*dt))
    if spin_echo == True:
        s_bar = np.sum((1/T * s*dt)*dd_window(len(s),1))
    return (s_bar,T)

def find_T_minimize(t0, f_chirp,d = 1e10):
    """
    Find cycle duration by minimizing the waveform difference.
    """
    def cycle_err(T_array):
        T = T_array[0]  # extract scalar from array
        cyc_start = h_chirped(t0, 1, f_chirp(t0), 0) - h_chirped(t0 + d/c, 1, f_chirp(t0 - d/c), 0)
        cyc_end   = h_chirped(t0 + T, 1, f_chirp(t0+T), 0) - h_chirped(t0 + T + d/c, 1, f_chirp(t0 + T - d/c), 0)
        return smooth_abs(cyc_start - cyc_end)
    T0 = np.array([1 / f_chirp(t0)])

def generate_signals(f0=0.01, N_measurements=1200, M_chirp=500, phi=0, t0=0, amp = 1e-20, fixed_T=False, optimized_T=True,d = 1e10,spin_echo = True):
    t = np.linspace(-1000, 240*3600, 100000)
    Mc = M_chirp * M0
    from scipy.interpolate import interp1d
    f = f_t(t, f0, Mc)
    f_chirp = interp1d(t, f)
    
    signals = []
    t0s1 = []
    Ts = []
    print(amp)
    for i in range(N_measurements):
        s_bar, T = signal_chirped(
            t0,
            H=amp,
            f_chirp=f_chirp,
            phi=phi,
            spin_echo=spin_echo,
            fixed_T=fixed_T,
            optimized_T=optimized_T,
            d = d
        )
        t0 += T
        t0s1.append(t0)
        signals.append(s_bar)
        Ts.append(T)
    
    return signals, Ts


def compute_complex_response(
    positions,
    arms,          # here: photon propagation directions n̂
    f_gw,          
    A_plus,
    A_cross,
    theta,
    phi_sky,
    psi=0
):
    # GW propagation vector
    k = -np.array([
        np.sin(theta) * np.cos(phi_sky),
        np.sin(theta) * np.sin(phi_sky),
        np.cos(theta)
    ])

    def make_perp_basis(k):
        k = k / np.linalg.norm(k)
        if np.abs(k[2]) < 0.99:
            tmp = np.array([0,0,1])
        else:
            tmp = np.array([1,0,0])
        ex = np.cross(tmp, k)
        ex /= np.linalg.norm(ex)
        ey = np.cross(k, ex)
        return ex, ey

    ex, ey = make_perp_basis(k)
    # Polarization tensors
    e_plus = np.outer(ex, ex) - np.outer(ey, ey)
    e_cross = np.outer(ex, ey) + np.outer(ey, ex)

    # Rotate by polarization angle psi
    e_plus_rot  = e_plus * np.cos(2 * psi) + e_cross * np.sin(2 * psi)
    e_cross_rot = -e_plus * np.sin(2 * psi) + e_cross * np.cos(2 * psi)

    # --- CLOCK ANTENNA GEOMETRY ---
    # n̂ = photon propagation direction
    nk = np.einsum("di,i->d", arms, k)
    denom = 1.0 - nk

    # Geometry-only detector tensor
    D = 0.5 * np.einsum("di,dj->dij", arms, arms) / denom[:, None, None]

    # Antenna factors
    F_plus  = np.einsum("dij,ij->d", D, e_plus_rot)
    F_cross = np.einsum("dij,ij->d", D, e_cross_rot)

    # Keep original phase convention
    tau = np.dot(positions, k) / c

    # Pure antenna-pattern response (no transfer function)
    S = (F_plus * A_plus + 1j*F_cross * A_cross) * np.exp(-1j * 2 * np.pi * f_gw * tau)
    
    

    return S, nk

def compute_fd_complex_response(
    positions,
    arm_lengths,
    arms,          # here: photon propagation directions n̂
    f_gw,          
    A_plus,
    A_cross,
    theta,
    phi_sky,
    psi=0
):
    arm_lengths = np.array(arm_lengths)
    # GW propagation vector
    
    k = -np.array([
        np.sin(theta) * np.cos(phi_sky),
        np.sin(theta) * np.sin(phi_sky),
        np.cos(theta)
    ])

    def make_perp_basis(k):
        k = k / np.linalg.norm(k)
        if np.abs(k[2]) < 0.99:
            tmp = np.array([0,0,1])
        else:
            tmp = np.array([1,0,0])
        ex = np.cross(tmp, k)
        ex /= np.linalg.norm(ex)
        ey = np.cross(k, ex)
        return ex, ey

    ex, ey = make_perp_basis(k)

    # Polarization tensors
    e_plus = np.outer(ex, ex) - np.outer(ey, ey)
    e_cross = np.outer(ex, ey) + np.outer(ey, ex)

    # Rotate by polarization angle psi
    e_plus_rot  = e_plus * np.cos(2 * psi) + e_cross * np.sin(2 * psi)
    e_cross_rot = -e_plus * np.sin(2 * psi) + e_cross * np.cos(2 * psi)

    # --- CLOCK ANTENNA GEOMETRY ---
    # n̂ = photon propagation direction
    nk = np.einsum("di,i->d", arms, k)
    denom = 1.0 - nk

    # Geometry-only detector tensor
    D = 0.5 * np.einsum("di,dj->dij", arms, arms) / denom[:, None, None]

    # Antenna factors
    F_plus  = np.einsum("dij,ij->d", D, e_plus_rot)
    F_cross = np.einsum("dij,ij->d", D, e_cross_rot)

    # Keep original phase convention
    tau = np.dot(positions, k) / c
    
    phase_delay = 2 * np.pi * f_gw * arm_lengths * denom / c
    transfer = 1.0 - np.exp(-1j * phase_delay)
    
    # Pure antenna-pattern response (no transfer function)
    S = (F_plus * A_plus + 1j*F_cross * A_cross) * np.exp(-1j * 2 * np.pi * f_gw * tau) * transfer
    
    

    return S, nk

def single_band_network_response(positions, arms, f_gw1, N_cycles_1, A_plus, A_cross, theta_sky, phi_sky,psi,M_chirp =500, arm_lengths = [1e10,1e10,1e10],n_ensembles = 10):
    
        
    S, nk = compute_complex_response(positions, arms, f_gw1 ,A_plus, A_cross, theta = theta_sky, phi_sky = phi_sky,psi = psi)
    eff_arm_lengths =np.array(arm_lengths)*(1-nk)


    fits = []
    sigs = []
    for s,d in zip(S,eff_arm_lengths):
        f_measured = []
        f_true = []
        f_err = []
        t_starts = np.linspace(0,1/(f_gw1),n_ensembles)
        for t0 in tqdm(t_starts):    
            signals,Ts = generate_signals(f0=f_gw1, N_measurements=N_cycles_1, M_chirp=M_chirp, t0 = t0, phi=np.angle(s), amp = np.abs(s)*5e-20,d = d)
            ffd_mean,ffd_err = clock_average_single_point(signals,Ts)    
            f_measured.append(ffd_mean)
            f_true.append(np.mean(signals))
            f_err.append(ffd_err)
        plt.errorbar(t_starts,f_measured,f_err,marker='o', markersize=6,elinewidth=1.5,capsize = 3,
                    linestyle='none')   
        fits.append(fit_sine_known_f(t_starts, f_measured, yerr=np.array(f_err),f = f_gw1))
        sigs.append(fit_sine_known_f(t_starts, f_true, yerr=np.array(f_err)/1e3, f = f_gw1))
    T_1 = np.sum(Ts)
    
    fit_S = []
    true_S =[]
    phase_err = []
    rel_amp_err = []
    for fit in fits:
            fit_S.append(fit['amplitude']*np.exp(1j*fit['phase']))
            phase_err.append(fit['phase_err'])
            rel_amp_err.append(np.abs(fit['amplitude_err']/fit['amplitude']))
    
    for sig in sigs:
            true_S.append(sig['amplitude']*np.exp(1j*sig['phase']))
            
    
    

    return fit_S,T_1,rel_amp_err,phase_err,true_S


  

In [33]:
##Source Setup

M_chirp  = 1000 ##In Solar Masses (Start at 500, go up to 2000)


##Natural_Polarization_Frame
inc_angle =np.pi/3

A_cross  = np.cos(inc_angle)
A_plus = (1+np.cos(inc_angle)**2)/2

##Rotating into detector frame 
psi  = np.pi/3 


##Sky Location
theta_sky  = 0.8
phi_sky  = 0.2




##Measurement Cycles
f_gw1 = 10/1000 ##Keep this in the 10s of mHz 
## Make a few data sets where the distinction 

#Network Setup
d1 = 1e10

n_ensembles = 5
theta_1 = 0
theta_2 = np.pi/6
theta_3 = np.pi/3

r = 1.46e11*4
positions = np.array([[r*np.cos(theta_1),r*np.sin(theta_1),0],[r*np.cos(theta_2),r*np.sin(theta_2),0],[r*np.cos(theta_3),r*np.sin(theta_3),0],[r*np.cos(theta_1),r*np.sin(theta_1),0],[r*np.cos(theta_2),r*np.sin(theta_2),0],[r*np.cos(theta_3),r*np.sin(theta_3),0]])
arms = np.array([[np.sin(theta_1),np.cos(theta_1),0],[np.sin(theta_2),np.cos(theta_2),0],[np.sin(theta_3),np.cos(theta_3),0],[-np.sin(theta_1),-np.cos(theta_1),0],[-np.sin(theta_2),-np.cos(theta_2),0],[-np.sin(theta_3),-np.cos(theta_3),0]])
arm_lengths = [1e10,1e10,1e10,1e10,1e10,1e10]


N_cycles = [10,50,100,500,1000,2000]




In [34]:
S, nk = compute_complex_response(positions, arms, f_gw1 ,A_plus, A_cross, theta = theta_sky, phi_sky = phi_sky,psi = psi)

In [35]:
S

array([-0.16639265+0.14058761j,  0.11537043+0.08263882j,
       -0.00928083+0.09912586j, -0.2217027 +0.18731988j,
        0.32409531+0.2321466j , -0.04874672+0.52064961j])

In [38]:
generate_signals(f0=f_gw1, N_measurements=2, M_chirp=M_chirp, t0 = 0, phi=np.angle(S[0]), amp = np.abs(S[0])*5e-20,d = 1e10)

1.0891669790347608e-20


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

In [41]:
np.abs(S)*5

array([1.08916698, 0.70956874, 0.4977969 , 1.45121351, 1.9933001 ,
       2.61463313])